# SHAP и CV-отбор признаков

Этот ноутбук отдельно считает отбор признаков для этапа предобработки. В `02_preprocessing.ipynb` итоговый список `selected_features` уже зафиксирован, а здесь показано, как его можно получить: подготовить данные, посчитать SHAP-важности через RandomForest и проверить разные top-N наборы признаков через 3-fold CV на Logistic Regression.

In [8]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import shap
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import fbeta_score, make_scorer, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd()
while ROOT.name != "NN-project" and ROOT.parent != ROOT:
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))
import utils

SEED = 42
RAW_PATH = ROOT / "data/raw/diabetic_data.csv"
RESULTS_DIR = ROOT / "results/feature_selection"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)

## 1. Повторяем базовую предобработку

Сначала повторяем те же шаги, что используются во втором этапе: оставляем первый визит пациента, удаляем смерть/хоспис, формируем `target`, убираем идентификаторы и признаки с большим количеством пропусков, группируем ICD-9 диагнозы и добавляем новые признаки. Это нужно, чтобы SHAP-отбор считался на той же логике данных, что и финальный preprocessing.

In [9]:
def map_icd9(code) -> str:
    if pd.isna(code) or str(code).strip() in ("", "?"):
        return "Other"

    code = str(code).strip()

    if code.startswith("250"):
        return "Diabetes"

    if code.startswith(("E", "V")):
        return "Other"

    try:
        num = float(code)
    except ValueError:
        return "Other"

    if 1 <= num <= 139:
        return "Infectious"
    if 140 <= num <= 239:
        return "Neoplasms"
    if 240 <= num <= 279:
        return "Endocrine"
    if 280 <= num <= 289:
        return "Blood"
    if 290 <= num <= 319:
        return "Mental"
    if 320 <= num <= 389:
        return "Nervous"
    if 390 <= num <= 459:
        return "Circulatory"
    if 460 <= num <= 519:
        return "Respiratory"
    if 520 <= num <= 579:
        return "Digestive"
    if 580 <= num <= 629:
        return "Genitourinary"
    if 630 <= num <= 679:
        return "Pregnancy"
    if 680 <= num <= 709:
        return "Skin"
    if 710 <= num <= 739:
        return "Musculoskeletal"
    if 740 <= num <= 759:
        return "Congenital"
    if 760 <= num <= 779:
        return "Perinatal"
    if 780 <= num <= 799:
        return "Symptoms"
    if 800 <= num <= 999:
        return "Injury"

    return "Other"


def prepare_data(path: str = RAW_PATH) -> pd.DataFrame:
    df = utils.load_data(path)

    df = df.sort_values("encounter_id").drop_duplicates(subset="patient_nbr", keep="first")
    dead_hospice = {11, 13, 14, 19, 20, 21}
    df = df[~df["discharge_disposition_id"].isin(dead_hospice)].copy()

    df["target"] = (df["readmitted"] == "<30").astype(int)
    df = df.drop(columns=["readmitted", "encounter_id", "patient_nbr"])
    df = df.drop(columns=["weight", "payer_code", "max_glu_serum", "A1Cresult"])

    df["medical_specialty"] = df["medical_specialty"].fillna("Unknown")
    df["race"] = df["race"].fillna("Unknown")
    df = df[df["gender"] != "Unknown/Invalid"].copy()

    for col in ["diag_1", "diag_2", "diag_3"]:
        df[col] = df[col].map(map_icd9)

    for col in ["admission_type_id", "discharge_disposition_id", "admission_source_id"]:
        df[col] = df[col].astype("Int64").astype("string")

    time = df["time_in_hospital"].clip(lower=1)
    df["total_prior_visits"] = df["number_outpatient"] + df["number_emergency"] + df["number_inpatient"]
    df["has_prior_visits"] = (df["total_prior_visits"] > 0).astype(int)
    df["has_inpatient"] = (df["number_inpatient"] > 0).astype(int)
    df["procedures_per_day"] = df["num_procedures"] / time
    df["med_lab_ratio"] = df["num_medications"] / (df["num_lab_procedures"] + 1)

    age_mid_map = {
        "[0-10)": 5,
        "[10-20)": 15,
        "[20-30)": 25,
        "[30-40)": 35,
        "[40-50)": 45,
        "[50-60)": 55,
        "[60-70)": 65,
        "[70-80)": 75,
        "[80-90)": 85,
        "[90-100)": 95,
    }
    df["age_mid"] = df["age"].map(age_mid_map).astype(float)
    return df.reset_index(drop=True)


df = prepare_data()
print(df.shape)
print(df["target"].mean())
df.head(3)

(69970, 50)
0.08970987566099757


,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,medical_specialty,num_lab_procedures,num_procedures,...,metformin-pioglitazone,change,diabetesMed,target,total_prior_visits,has_prior_visits,has_inpatient,procedures_per_day,med_lab_ratio,age_mid
0,Caucasian,Female,[80-90),2,1,4,13,Unknown,68,2,...,No,Ch,Yes,0,0,0,0,0.153846,0.405797,85.0
1,Caucasian,Female,[90-100),3,3,4,12,InternalMedicine,33,3,...,No,Ch,Yes,0,0,0,0,0.250000,0.529412,95.0
2,Caucasian,Male,[40-50),1,1,7,1,Unknown,51,0,...,No,Ch,Yes,0,0,0,0,0.000000,0.153846,45.0


## 2. Кандидаты на отбор

Здесь задаем признаки, из которых выбирается финальный набор. В кандидаты входят исходные числовые признаки, созданные признаки и категориальные признаки после очистки. Для лекарств берем все колонки, где значения показывают режим приема препарата.

In [10]:
numeric_candidates = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
    "total_prior_visits",
    "has_prior_visits",
    "has_inpatient",
    "procedures_per_day",
    "med_lab_ratio",
    "age_mid",
]

categorical_candidates = [
    "race",
    "gender",
    "age",
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
    "medical_specialty",
    "diag_1",
    "diag_2",
    "diag_3",
    "change",
    "diabetesMed",
]

drug_features = [
    "metformin",
    "repaglinide",
    "nateglinide",
    "chlorpropamide",
    "glimepiride",
    "acetohexamide",
    "glipizide",
    "glyburide",
    "tolbutamide",
    "pioglitazone",
    "rosiglitazone",
    "acarbose",
    "miglitol",
    "troglitazone",
    "tolazamide",
    "examide",
    "citoglipton",
    "insulin",
    "glyburide-metformin",
    "glipizide-metformin",
    "glimepiride-pioglitazone",
    "metformin-rosiglitazone",
    "metformin-pioglitazone",
  ]
categorical_candidates = categorical_candidates + drug_features

candidate_features = numeric_candidates + categorical_candidates
print("numeric:", len(numeric_candidates))
print("categorical:", len(categorical_candidates))
print("all candidates:", len(candidate_features))

numeric: 14
categorical: 35
all candidates: 49


## 3. RandomForest + SHAP

Для SHAP сначала обучаем RandomForest на подготовленных признаках. Категориальные признаки кодируются через one-hot, числовые заполняются медианой. SHAP считается на подвыборке, чтобы расчет не был слишком тяжелым. Потом важности one-hot колонок суммируются обратно до исходных признаков.

In [11]:
X = df[candidate_features]
y = df["target"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_candidates),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_candidates),
    ],
    verbose_feature_names_out=True,
)

sample_n = min(12000, len(df))
sample_idx = df.sample(sample_n, random_state=SEED).index
X_sample = X.loc[sample_idx]
y_sample = y.loc[sample_idx]

X_rf = preprocessor.fit_transform(X_sample)
encoded_feature_names = preprocessor.get_feature_names_out()

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=20,
    class_weight="balanced",
    n_jobs=-1,
    random_state=SEED,
)
rf.fit(X_rf, y_sample)

explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_rf)

if isinstance(shap_values, list):
    class_1_shap = shap_values[1]
else:
    class_1_shap = shap_values[:, :, 1] if shap_values.ndim == 3 else shap_values

encoded_importance = pd.Series(np.abs(class_1_shap).mean(axis=0), index=encoded_feature_names)

def original_feature_name(encoded_name: str) -> str:
    if encoded_name.startswith("num__"):
        return encoded_name.replace("num__", "", 1)
    if encoded_name.startswith("cat__"):
        raw = encoded_name.replace("cat__", "", 1)
        for feature in sorted(categorical_candidates, key=len, reverse=True):
            if raw == feature or raw.startswith(feature + "_"):
                return feature
    return encoded_name

feature_importance = (
    encoded_importance.rename_axis("encoded_feature")
    .reset_index(name="mean_abs_shap")
    .assign(feature=lambda d: d["encoded_feature"].map(original_feature_name))
    .groupby("feature", as_index=False)["mean_abs_shap"].sum()
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)

feature_importance.to_csv(RESULTS_DIR / "shap_feature_importance.csv", index=False)
feature_importance.head(30)

,feature,mean_abs_shap
0,discharge_disposition_id,0.035212
1,number_inpatient,0.011423
2,has_inpatient,0.011188
3,time_in_hospital,0.009412
4,procedures_per_day,0.008960
5,age,0.008858
6,age_mid,0.007660
7,diag_3,0.007199
8,has_prior_visits,0.006356
9,diag_1,0.006112


## 4. Проверка top-N признаков через CV

SHAP дает ранжирование признаков, но сам по себе не доказывает, сколько признаков нужно оставить. Поэтому проверяем несколько вариантов `top-N` через 3-fold CV. Проверку делаем на Logistic Regression: это простой baseline, по которому удобно понять, сколько признаков дают устойчивый сигнал. Для оценки используем ROC-AUC и F2, потому что в задаче важен не только общий ранжирующий сигнал, но и поиск пациентов с риском ранней повторной госпитализации.

In [12]:
def build_preprocessor(features: list[str]) -> ColumnTransformer:
    num = [f for f in features if f in numeric_candidates]
    cat = [f for f in features if f in categorical_candidates]
    return ColumnTransformer(
        transformers=[
            ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num),
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
        ]
    )


f2_scorer = make_scorer(fbeta_score, beta=2, zero_division=0)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
ordered_features = feature_importance["feature"].tolist()

rows = []
for top_n in [10, 12, 15, 20, 25, 30, len(ordered_features)]:
    features = ordered_features[:top_n]
    logreg_model = Pipeline(
        steps=[
            ("prep", build_preprocessor(features)),
            ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", n_jobs=-1, random_state=SEED)),
        ]
    )
    logreg_scores = cross_validate(
        logreg_model,
        df[features],
        y,
        cv=cv,
        scoring={"roc_auc": "roc_auc", "f2": f2_scorer},
        n_jobs=-1,
    )
    rows.append(
        {
            "top_n": top_n,
            "features": features,
            "logreg_3fold_roc_auc_mean": logreg_scores["test_roc_auc"].mean(),
            "logreg_3fold_roc_auc_std": logreg_scores["test_roc_auc"].std(),
            "logreg_3fold_f2_mean": logreg_scores["test_f2"].mean(),
            "logreg_3fold_f2_std": logreg_scores["test_f2"].std(),
        }
    )

cv_results = pd.DataFrame(rows).sort_values("logreg_3fold_f2_mean", ascending=False)
cv_results.to_csv(RESULTS_DIR / "cv_shap_topn_results.csv", index=False)
cv_results[["top_n", "logreg_3fold_roc_auc_mean", "logreg_3fold_f2_mean"]]

,top_n,logreg_3fold_roc_auc_mean,logreg_3fold_f2_mean
4,25,0.644115,0.338860
3,20,0.644681,0.337883
2,15,0.644314,0.337510
5,30,0.643660,0.337269
6,49,0.643259,0.336780
0,10,0.643440,0.334859
1,12,0.641088,0.334769


## 5. Финальный список признаков

После проверки выбираем вариант, который дает хороший баланс между качеством и простотой. Если лучший `top-N` отличается от текущего списка в `02_preprocessing.ipynb`, нужно решить, обновлять ли preprocessing или оставить прежний стабильный набор.

In [13]:
best = cv_results.iloc[0]
selected_features = best["features"]
numeric_selected = [f for f in selected_features if f in numeric_candidates]
categorical_selected = [f for f in selected_features if f in categorical_candidates]

selection_report = {
    "method": "RandomForest SHAP feature ranking + LogisticRegression 3-fold CV top-N check",
    "best_top_n": int(best["top_n"]),
    "logreg_3fold_roc_auc_mean": float(best["logreg_3fold_roc_auc_mean"]),
    "logreg_3fold_f2_mean": float(best["logreg_3fold_f2_mean"]),
    "numeric": numeric_selected,
    "categorical": categorical_selected,
    "selected_features": selected_features,
}

with open(RESULTS_DIR / "shap_selection_report.json", "w", encoding="utf-8") as f:
    json.dump(selection_report, f, indent=2, ensure_ascii=False)

print("best top_n:", selection_report["best_top_n"])
print("LogReg ROC-AUC:", round(selection_report["logreg_3fold_roc_auc_mean"], 6))
print("LogReg F2:", round(selection_report["logreg_3fold_f2_mean"], 6))
print("numeric:", numeric_selected)
print("categorical:", categorical_selected)

best top_n: 25
LogReg ROC-AUC: 0.644115
LogReg F2: 0.33886
numeric: ['number_inpatient', 'has_inpatient', 'time_in_hospital', 'procedures_per_day', 'age_mid', 'has_prior_visits', 'number_diagnoses', 'num_lab_procedures', 'total_prior_visits', 'med_lab_ratio', 'num_medications', 'number_outpatient']
categorical: ['discharge_disposition_id', 'age', 'diag_3', 'diag_1', 'diag_2', 'medical_specialty', 'insulin', 'diabetesMed', 'admission_source_id', 'admission_type_id', 'metformin', 'glipizide', 'change']
